# Анализ зарплат

Анализ распределения экономических статистик среди направлений.     

Проверяемые статистические гипотезы:    
1. Выпускники различных категорий направлений получаются значимо различающиеся медианные зарплаты   
2. STEM-направления в среднем зарабатывают больше Not-STEM   
3. Выпускники каждого направления будут уверены в диапазоне своей зарплаты      
4. Уровень безработицы и доля занятых полный рабочий день взаимосвязаны     
5. Медианный доход и доля занятых полный рабочий день взаимосвязаны     
6. Уровень безработицы связан с медианным доходом выпускников   
7. Массовость специальности связана с медианным доходом выпускников

## Загрузка библиотек и установка настроек


In [2]:
import pandas as pd
import numpy as np
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 50)


## Введение констант


In [3]:
PROCESSED_DATA_DIR = Path("../data/processed")


## Загрузка набора данных


In [4]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "majors_all_ages_analytics.parquet")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   Major_code                     172 non-null    int64   
 1   Major                          172 non-null    category
 2   Major_category                 172 non-null    category
 3   Total                          172 non-null    int64   
 4   Employed                       172 non-null    int64   
 5   Employed_full_time_year_round  172 non-null    int64   
 6   Unemployed                     172 non-null    int64   
 7   Unemployment_rate              172 non-null    float64 
 8   Median                         172 non-null    int64   
 9   P25th                          172 non-null    int64   
 10  P75th                          172 non-null    float64 
 11  full_time_rate                 172 non-null    float64 
 12  salary_range_rate              172 n

## Гипотезы

### Выпускники различных категорий направлений получаются значимо различающиеся медианные зарплаты

$H_0$ - `Median` по группам `Major_category` не различаются;    
$H_1$ - `Median` по группам `Major_category` различаются.

In [5]:
order = (
    df.groupby("Major_category", observed=True)["Median"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.box(
    df,
    y="Major_category",
    x="Median",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={"Median": "Медианная зарплата", "Major_category": ""},
    title="<b>Медианные зарплаты по категориям специальностей</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для критерия Краскела–Уоллиса

**Независимость наблюдений** - каждое направление принадлежит ровно одной категории.    
**Тип данных** - медианная зарплата является непрерывной переменной.    
**Форма распределений** - кардинально разных форм между группами не замечается. 

Следовательно, критерий Краскела-Уоллиса применим.


#### Тест

In [6]:
groups = [g["Median"].values for _, g in df.groupby("Major_category", observed=True)]
h_stat, p_val = stats.kruskal(*groups)

print(f"Краскел–Уоллис: p = {p_val}")


Краскел–Уоллис: p = 1.3054915777861272e-18


#### Размер эффекта

In [7]:
n = sum(len(g) for g in groups)
k = len(groups)
eta_sq = (h_stat - k + 1) / (n - k)

print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.6648


#### Вывод

Группы направлений статистически значимо отличаются по медианной зарплате. Размер эффекта η² подтверждает практическую значимость этих различий.  
Наибольшую медианную зарплату получают выпускники `Engineering`, а наименьшую - `Education`.

### STEM-направления зарабатывают больше Not-STEM

$H_0$ - медианные зарплаты выпускников STEM и Not-STEM направлений не различаются;   
$H_1$ - выпускники STEM-направлений зарабатывают больше, чем выпускники Not-STEM.

In [ ]:
stem_categories = ["Engineering", "Computers & Mathematics", "Physical Sciences", "Biology & Life Science"]
df["is_stem"] = df["Major_category"].isin(stem_categories).map({True: "STEM", False: "Not STEM"})

order = ["Not STEM", "STEM"]
color_map = {"STEM": "#1f9e89", "Not STEM": "#440154"}

fig = px.box(
    df,
    y="is_stem",
    x="Median",
    color="is_stem",
    category_orders={"is_stem": order},
    color_discrete_map=color_map,
    labels={"Median": "Медианная зарплата", "is_stem": ""},
    title="<b>Медианные зарплаты: STEM vs Not STEM</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=700,
    height=300,
)
fig.show()


#### Предусловия для критерия Манна–Уитни

**Независимость наблюдений** - каждое направление принадлежит ровно одной группе (STEM или Not STEM).    
**Тип данных** - медианная зарплата является непрерывной переменной.    
**Форма распределений** - кардинально разных форм между группами не замечается.   

Следовательно, критерий Манна–Уитни применим.

#### Тест

In [ ]:
stem = df.loc[df["is_stem"] == "STEM", "Median"].values
not_stem = df.loc[df["is_stem"] == "Not STEM", "Median"].values

u_stat, p_val = stats.mannwhitneyu(stem, not_stem, alternative="greater")

print(f"Манн–Уитни: U = {u_stat:.0f}, p = {p_val:.4f}")


#### Размер эффекта

In [ ]:
n1, n2 = len(stem), len(not_stem)
r = (2 * u_stat) / (n1 * n2) - 1

abs_r = abs(r)
if abs_r < 0.1:
    magnitude = "тривиальный"
elif abs_r < 0.3:
    magnitude = "слабый"
elif abs_r < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта r = {r:.4f} ({magnitude} по Cohen)")


#### Вывод

Нулевая гипотеза отвергается (p < 0.05): STEM-направления статистически значимо зарабатывают больше Not-STEM.   
Размер эффекта большой — различие между группами является практически значимым.

### Выпускники каждого направления будут уверены в диапазоне своей зарплаты

$H_0$ - дисперсии `Median` по группам `Major_category` равны;   
$H_1$ - дисперсии `Median` по группам `Major_category` различаются.

In [8]:
order = (
    df.groupby("Major_category", observed=True)["Median"]
    .std()
    .sort_values(ascending=True)
    .index.tolist()
)

std_df = (
    df.groupby("Major_category", observed=True)["Median"]
    .std()
    .reset_index()
    .rename(columns={"Median": "Salary_std"})
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.bar(
    std_df,
    y="Major_category",
    x="Salary_std",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={"Salary_std": "Стандартное отклонение медианной зарплаты", "Major_category": ""},
    title="<b>Разброс зарплат внутри категорий специальностей</b>",
    template="plotly_white",
    orientation="h",
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для теста Левена

**Независимость наблюдений** - каждое направление принадлежит ровно одной категории.    
**Тип данных** - медианная зарплата является непрерывной переменной.    
**Устойчивость к ненормальности** - тест Левена применим при умеренных отклонениях от нормальности. 

Следовательно, тест Левена применим.

#### Тест

In [9]:
groups = [g["Median"].values for _, g in df.groupby("Major_category", observed=True)]
levene_stat, p_val = stats.levene(*groups)

print(f"Тест Левена: p = {p_val}")


Тест Левена: p = 0.019136790704559533


#### Размер эффекта

In [10]:
k = len(groups)
n = sum(len(g) for g in groups)
eta_sq = (levene_stat * (k - 1)) / (levene_stat * (k - 1) + (n - k))

print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.1529


#### Вывод

Обнаружены значимые различия в дисперсии зарплат между категориями направлений (p = 0.019).     
Размер эффекта η² = 0.15 подтверждает практическую значимость - разброс зарплат внутри категорий существенно различается. Выпускники разных категорий имеют разный уровень уверенности в своей зарплате: наиболее предсказуемы зарплаты в `Communications & Journalism` и `Arts`, наименее - в `Engineering` и `Health`.

### Уровень безработицы и доля занятых полный рабочий день взаимосвязаны

$H_0$ - `Unemployment_rate` и `full_time_rate` не коррелируют;  
$H_1$ - между `Unemployment_rate` и `full_time_rate` существует монотонная связь.

In [11]:
x_col, y_col = "full_time_rate", "Unemployment_rate"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Unemployment_rate vs full_time_rate</b>", font_size=13),
    xaxis_title="Доля занятых полный рабочий день (full_time_rate)",
    yaxis_title="Уровень безработицы (Unemployment_rate)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** - каждое направление является самостоятельной единицей.  
**Тип данных** - обе переменные непрерывны.  
**Монотонность** - диаграмма рассеяния не показывает явной немонотонной зависимости.  
**Нормальность** - для долей (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен ранговый критерий Спирмена, не требующий нормального распределения.

Следовательно, критерий Спирмена применим.

#### Тест

In [12]:
rho, p_val = stats.spearmanr(df["full_time_rate"], df["Unemployment_rate"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = -0.2615, p = 0.0005


#### Размер эффекта

In [13]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude} по Cohen)")


Размер эффекта |ρ| = 0.2615 (слабый по Cohen)


#### Вывод

Нулевая гипотеза об отсутствии корреляции отвергается (p = 0.0005): между `full_time_rate` и `Unemployment_rate` существует статистически значимая отрицательная связь (ρ = −0.26).     
Размер эффекта малый, поэтому практическая значимость ограничена - направления с более высокой долей занятых полный день лишь незначительно коррелированны с более низкой безработицей.

### Медианный доход и доля занятых полный рабочий день взаимосвязаны

$H_0$ - `Median` и `full_time_rate` не коррелируют;     
$H_1$ - между `Median` и `full_time_rate` существует монотонная связь.

In [14]:
x_col, y_col = "Median", "full_time_rate"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Median vs full_time_rate</b>", font_size=13),
    xaxis_title="Медианная зарплата (Median)",
    yaxis_title="Доля занятых полный рабочий день (full_time_rate)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** - каждое направление является самостоятельной единицей.  
**Тип данных** - обе переменные непрерывны.  
**Монотонность** - диаграмма рассеяния не показывает явной немонотонной зависимости.  
**Нормальность** - для долей (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен ранговый критерий Спирмена, не требующий нормального распределения.

Следовательно, критерий Спирмена применим.

#### Тест

In [15]:
rho, p_val = stats.spearmanr(df["Median"], df["full_time_rate"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = 0.7553, p = 0.0000


#### Размер эффекта

In [16]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude} по Cohen)")


Размер эффекта |ρ| = 0.7553 (большой по Cohen)


#### Вывод

Нулевая гипотеза об отсутствии корреляции отвергается (p $\approx$ 0): между `Median` и `full_time_rate` существует статистически значимая положительная связь (ρ = 0.76).      
Размер эффекта большой - направления с более высокой медианной зарплатой значительно чаще характеризуются высокой долей занятых на полный рабочий день.

### Уровень безработицы связан с медианным доходом выпускников

$H_0$ - `Unemployment_rate` и `Median` не коррелируют;      
$H_1$ - между `Unemployment_rate` и `Median` существует отрицательная монотонная связь.

In [17]:
x_col, y_col = "Unemployment_rate", "Median"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Median vs Unemployment_rate</b>", font_size=13),
    xaxis_title="Уровень безработицы (Unemployment_rate)",
    yaxis_title="Медианная зарплата (Median)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** - каждое направление является самостоятельной единицей.    
**Тип данных** - обе переменные непрерывны.     
**Монотонность** - диаграмма рассеяния не показывает явной немонотонной зависимости.    
**Нормальность** - для доли (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен ранговый критерий Спирмена, не требующий нормального распределения.   

Следовательно, критерий Спирмена применим.

#### Тест

In [18]:
rho, p_val = stats.spearmanr(df["Unemployment_rate"], df["Median"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = -0.2914, p = 0.0001


#### Размер эффекта

In [19]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude} по Cohen)")


Размер эффекта |ρ| = 0.2914 (слабый по Cohen)


#### Вывод

Нулевая гипотеза об отсутствии корреляции отвергается (p = 0.0001): между `Unemployment_rate` и `Median` существует статистически значимая отрицательная связь (ρ = −0.29). Размер эффекта слабый - направления с более высоким уровнем безработицы незначительно ассоциированы с более низкими медианными зарплатами.

### Массовость специальности связана с медианным доходом выпускников

$H_0$ - `Total` и `Median` не коррелируют;      
$H_1$ - между `Total` и `Median` существует отрицательная монотонная связь.

In [28]:
x_col, y_col = "Total", "Median"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Median vs Total</b>", font_size=13),
    xaxis_title="Общее число выпускников (Total)",
    yaxis_title="Медианная зарплата (Median)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** - каждое направление является самостоятельной единицей.    
**Тип данных** - обе переменные непрерывны.     
**Монотонность** - диаграмма рассеяния не показывает явной немонотонной зависимости.       
**Нормальность** - распределение числа выпускников скошено, поэтому предпочтителен ранговый критерий Спирмена, не требующий нормального распределения.      

Следовательно, критерий Спирмена применим.

#### Тест

In [21]:
rho, p_val = stats.spearmanr(df["Total"], df["Median"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = -0.1302, p = 0.0886


#### Вывод

Нулевая гипотеза об отсутствии корреляции не отвергается (p = 0.089): связь между `Total` и `Median` статистически незначима.   
Данные не подтверждают гипотезу о зависимости между массовостью специальности и медианным доходом.

## Выводы

Проверяемые статистические гипотезы:    
1. Выпускники различных категорий направлений получают значимо различающиеся медианные зарплаты   
  -> Существует неравенство в распределении зарплат между направлениями

2. STEM-направления в среднем зарабатывают больше Not-STEM   
  -> STEM-направления статистически значимо зарабатывают больше Not-STEM

3. Выпускники каждого направления будут уверены в диапазоне своей зарплаты      
  -> Существуют направления, выпускники которых будут иметь сильно отличающиеся зарплаты

4. Уровень безработицы и доля занятых полный рабочий день взаимосвязаны     
  -> Направления с более высокой долей занятых полный день лишь незначительно коррелированны с более низкой безработицей

5. Медианный доход и доля занятых полный рабочий день взаимосвязаны     
  -> Чем больше медианный доход, тем выше доля занятости полный день, и наоборот

6. Уровень безработицы связан с медианным доходом выпускников   
  -> Имеется слабая корреляция между уровнем безработицы и медианным доходом
  
7. Массовость специальности связана с медианным доходом выпускников     
  -> Количество выпускников направления не коррелирует с медианным доходом